# Computer Lab 1: Time series analysis and statistical properties of asset returns

## Step 0 — Setup and data

The Excel file has two simulated series, `et` and `ut`, for $t = 1,\dots,1000$.
Both are **shocks**, not returns yet. In Exercise 1 we build returns from `et`;
in Exercise 2 we treat `ut` itself as the return series.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import plotly.graph_objects as go
import plotly.io as pio
import kaleido
kaleido.get_chrome_sync()

# One shared look for every chart in the notebook
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8a85"
pio.templates["lab"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, sans-serif", size=13, color="#3a3a37"),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="#c3c2b7", ticks="outside", tickcolor="#c3c2b7"),
    yaxis=dict(gridcolor="#ecebe6", zerolinecolor="#c3c2b7"),
    hoverlabel=dict(bgcolor="white", font_size=12),
    margin=dict(l=60, r=20, t=60, b=50),
))
pio.templates.default = "lab"

# The notebook lives in ml-course/Empirical Finance, the data in ml-course/data
df = pd.read_excel("../data/DataLab1_2026.xlsx", sheet_name="Data", index_col="time")
df.head()

,et,ut
time,,
1,0.203622,-0.048721
2,-1.878116,0.497421
3,0.365329,0.528308
4,-1.980843,0.349441
5,-0.197768,-1.435491


In [2]:
# Quick look: mean, standard deviation, min/max of each shock series
df.describe().T

,count,mean,std,min,25%,50%,75%,max
et,1000.0,0.001899,1.017079,-3.193854,-0.672906,0.006578,0.679973,3.487221
ut,1000.0,0.057732,1.092629,-6.779732,-0.557402,0.092047,0.737728,4.126398


## Step 1 — Build the MA(3) log returns

$$r_t = 0.1 + 0.3\,e_{t-1} - 0.2\,e_{t-2} + 0.1\,e_{t-3} + e_t$$

Today's return is today's shock plus a weighted echo of the last three shocks.
`shift(k)` moves the series down $k$ rows, so row $t$ of `e_lag{k}` holds $e_{t-k}$.

The first three rows have no $e_{t-3}$ (it would be $e_0, e_{-1}, e_{-2}$), so they become
`NaN` and we drop them. That leaves $T = 997$ usable returns.

In [3]:
mu, theta = 0.1, [0.3, -0.2, 0.1]   # theta_1, theta_2, theta_3

ex1 = pd.DataFrame({"e": df["et"]})
for k in range(1, 4):
    ex1[f"e_lag{k}"] = ex1["e"].shift(k)

ex1["r"] = mu + theta[0]*ex1["e_lag1"] + theta[1]*ex1["e_lag2"] + theta[2]*ex1["e_lag3"] + ex1["e"]
ex1.head(6)   # rows 1-3 have NaN: that's the 3 lost observations

,e,e_lag1,e_lag2,e_lag3,r
time,,,,,
1,0.203622,NaN,NaN,NaN,NaN
2,-1.878116,0.203622,NaN,NaN,NaN
3,0.365329,-1.878116,0.203622,NaN,NaN
4,-1.980843,0.365329,-1.878116,0.203622,-1.375259
5,-0.197768,-1.980843,0.365329,-1.878116,-0.952899
6,0.033015,-0.197768,-1.980843,0.365329,0.506386


In [4]:
r = ex1["r"].dropna()
T = len(r)
print(f"T = {T} observations")
print(f"Sample mean of r: {r.mean():.4f}   (theory: mu = 0.1)")
print(f"Sample variance of r: {r.var():.4f}   (theory: sigma_e^2 * (1 + 0.3^2 + 0.2^2 + 0.1^2) = {df['et'].var()*(1+0.09+0.04+0.01):.4f})")

T = 997 observations
Sample mean of r: 0.1043   (theory: mu = 0.1)
Sample variance of r: 1.1747   (theory: sigma_e^2 * (1 + 0.3^2 + 0.2^2 + 0.1^2) = 1.1793)


In [5]:
fig = go.Figure()
fig.add_scatter(x=r.index, y=r, mode="lines", line=dict(color=BLUE, width=1),
                name="r_t", hovertemplate="t = %{x}<br>r_t = %{y:.3f}<extra></extra>")
fig.add_hline(y=r.mean(), line=dict(color=GREY, dash="dash", width=1))
fig.update_layout(title=f"Simulated MA(3) log returns r<sub>t</sub><br><sup>dashed line: sample mean = {r.mean():.3f}</sup>",
                  xaxis_title="t", yaxis_title="r<sub>t</sub>", height=320, showlegend=False)
fig.show()

## Step 2 — Exercise 1(a): Is $r_t$ normally distributed?

**Expectation.** $r_t$ is a linear combination of normal shocks, so if $e_t$ is normal, $r_t$ is normal too:
skewness $= 0$ and excess kurtosis $= 0$.

**What the two moments measure** (on the standardized variable $z = (r-\bar r)/s$):
- **Skewness** $= \text{mean}(z^3)$: asymmetry. Cubing keeps the sign, so a long left tail makes it negative.
- **Excess kurtosis** $= \text{mean}(z^4) - 3$: tail heaviness. A normal has $\text{mean}(z^4)=3$, so we subtract 3; positive means fatter tails than normal.

**Tests** ($H_0$: normal distribution):

$$Z_S = \sqrt{T/6}\cdot S \sim N(0,1), \qquad Z_K = \sqrt{T/24}\cdot K \sim N(0,1)$$
$$JB = \tfrac{T}{6}S^2 + \tfrac{T}{24}K^2 = Z_S^2 + Z_K^2 \sim \chi^2_2$$

JB is just the sum of the two squared $Z$'s, which is why it has 2 degrees of freedom.

In [6]:
def normality_tests(x, name):
    T = len(x)
    S = stats.skew(x)        # sample skewness
    K = stats.kurtosis(x)    # EXCESS kurtosis (scipy subtracts 3 by default)
    Z_S = np.sqrt(T/6) * S
    Z_K = np.sqrt(T/24) * K
    JB = Z_S**2 + Z_K**2
    return pd.DataFrame({
        "statistic":  [S, K, JB],
        "test stat":  [Z_S, Z_K, JB],
        "p-value":    [2*(1-stats.norm.cdf(abs(Z_S))), 2*(1-stats.norm.cdf(abs(Z_K))), 1-stats.chi2.cdf(JB, df=2)],
    }, index=[f"{name}: skewness (Z)", f"{name}: excess kurtosis (Z)", f"{name}: Jarque-Bera (chi2, 2 df)"])

print(f"T = {len(r)}")
normality_tests(r, "r").round(4)

T = 997


,statistic,test stat,p-value
r: skewness (Z),0.0260,0.3355,0.7372
r: excess kurtosis (Z),-0.1818,-1.1719,0.2412
"r: Jarque-Bera (chi2, 2 df)",1.4860,1.4860,0.4757


In [7]:
def histogram_vs_normal(x, name, nbins=40):
    """Histogram (as a density) with the normal curve that has the same mean and sd."""
    grid = np.linspace(x.min(), x.max(), 300)
    fig = go.Figure()
    fig.add_histogram(x=x, nbinsx=nbins, histnorm="probability density", name=name,
                      marker=dict(color=BLUE, line=dict(color="white", width=1)), opacity=0.85,
                      hovertemplate="bin %{x}<br>density %{y:.3f}<extra></extra>")
    fig.add_scatter(x=grid, y=stats.norm.pdf(grid, x.mean(), x.std()), mode="lines",
                    line=dict(color=ORANGE, width=2), name="Normal, same mean & sd",
                    hovertemplate="x = %{x:.2f}<br>normal density %{y:.3f}<extra></extra>")
    fig.update_layout(title=f"Distribution of {name}", xaxis_title=name, yaxis_title="density",
                      height=380, bargap=0, legend=dict(x=0.99, xanchor="right", y=0.99))
    fig.show()

# Visual check: does the histogram follow the normal curve?
histogram_vs_normal(r, "r<sub>t</sub>")

## Step 3 — Exercise 1(b): Sample ACF, individual tests, Ljung–Box

**Expectation.** In an MA(3), $r_t$ and $r_{t-k}$ share shocks only when $k \le 3$
(e.g. $r_t$ contains $e_{t-1}$, and so does $r_{t-1}$). For $k \ge 4$ they share no shocks,
so the *theoretical* autocorrelation is exactly zero.

**Sample ACF.** Following the lab instructions, $\hat\rho_k$ is the correlation (`corr`) between $r_t$ and
$r_{t-k}$ over the overlapping observations. (The textbook formula $\hat\gamma_k/\hat\gamma_0$ gives
almost the same numbers; it is shown for comparison.)

**Individual test**, $H_0: \rho_k = 0$: under independence $\hat\rho_k \approx N(0, 1/T)$, so
$z_k = \sqrt{T}\,\hat\rho_k \sim N(0,1)$. 95% CI: $\hat\rho_k \pm 1.96/\sqrt{T}$.

**Ljung–Box**, $H_0: \rho_1 = \dots = \rho_m = 0$ jointly:
$$Q_m = T(T+2)\sum_{k=1}^{m}\frac{\hat\rho_k^2}{T-k} \sim \chi^2_m$$
It squares the autocorrelations, so positive and negative ones both add evidence against $H_0$.

In [8]:
def acf_tests(x, name, max_lag=5):
    x = pd.Series(np.asarray(x))
    T = len(x)
    rows = []
    for k in range(1, max_lag + 1):
        rho = x.corr(x.shift(k))                      # lab method: CORREL(r_t, r_{t-k})
        xd = x - x.mean()
        rho_formula = (xd * xd.shift(k)).sum() / (xd**2).sum()   # gamma_k / gamma_0, eq. (29)-(31)
        z = np.sqrt(T) * rho
        rows.append({"lag": k, "rho_hat": rho, "rho (gamma_k/gamma_0)": rho_formula,
                     "z = sqrt(T)*rho": z, "p-value": 2*(1-stats.norm.cdf(abs(z))),
                     "CI low": rho - 1.96/np.sqrt(T), "CI high": rho + 1.96/np.sqrt(T)})
    tab = pd.DataFrame(rows).set_index("lag")
    Q = T*(T+2) * sum(tab.loc[k, "rho_hat"]**2 / (T-k) for k in tab.index)
    lb = pd.Series({"Q": Q, "df": max_lag, "p-value": 1 - stats.chi2.cdf(Q, df=max_lag)}, name=f"Ljung-Box {name}")
    return tab, lb

acf_r, lb_r = acf_tests(r, "r")
print(f"T = {len(r)},  1.96/sqrt(T) = {1.96/np.sqrt(len(r)):.4f}")
acf_r.round(4)

T = 997,  1.96/sqrt(T) = 0.0621


,rho_hat,rho (gamma_k/gamma_0),z = sqrt(T)*rho,p-value,CI low,CI high
lag,,,,,,
1,0.2229,0.2224,7.0369,0.0000,0.1608,0.2849
2,-0.1110,-0.1107,-3.5043,0.0005,-0.1731,-0.0489
3,0.0882,0.0879,2.7849,0.0054,0.0261,0.1503
4,0.0207,0.0206,0.6525,0.5141,-0.0414,0.0827
5,-0.0032,-0.0031,-0.0995,0.9207,-0.0652,0.0589


In [9]:
lb_r.round(4)

Q          70.2292
df          5.0000
p-value     0.0000
Name: Ljung-Box r, dtype: float64

In [10]:
# Cross-check with statsmodels' built-in Ljung-Box (uses the gamma_k/gamma_0 ACF, so tiny differences)
from statsmodels.stats.diagnostic import acorr_ljungbox
acorr_ljungbox(r, lags=[5])

,lb_stat,lb_pvalue
5,69.871105,1.090078e-13


In [11]:
def correlogram(tab, T, title, col="rho_hat"):
    """Bars = sample (P)ACF, whiskers = 95% CI around each estimate, dashed = +-1.96/sqrt(T) around zero."""
    band = 1.96 / np.sqrt(T)
    sig = (tab["CI low"] > 0) | (tab["CI high"] < 0) if "CI low" in tab else tab[col].abs() > band
    fig = go.Figure()
    fig.add_bar(x=tab.index, y=tab[col], width=0.45,
                marker=dict(color=[BLUE if s else "#b7d3f6" for s in sig]),
                error_y=dict(type="constant", value=band, color="#3a3a37", thickness=1.2, width=6),
                customdata=np.c_[tab[col] - band, tab[col] + band, np.sqrt(T) * tab[col]],
                hovertemplate=("lag %{x}<br>estimate = %{y:.4f}<br>95% CI [%{customdata[0]:.3f}, "
                               "%{customdata[1]:.3f}]<br>z = %{customdata[2]:.2f}<extra></extra>"),
                name="estimate")
    for s in (1, -1):
        fig.add_hline(y=s*band, line=dict(color=ORANGE, dash="dash", width=1))
    fig.add_hline(y=0, line=dict(color="#3a3a37", width=1))
    fig.update_layout(title=f"{title}<br><sup>dark bars significant at 5% · dashed lines: ±1.96/√T = ±{band:.3f}</sup>",
                      xaxis=dict(title="lag k", tickmode="linear", dtick=1),
                      yaxis_title="autocorrelation", height=380, showlegend=False)
    fig.show()

correlogram(acf_r, len(r), "Correlogram of r<sub>t</sub>")

## Step 4 — Exercise 1(c): Partial autocorrelations (PACF)

**What the PACF measures.** The ACF at lag $k$ is the *total* correlation between $r_t$ and $r_{t-k}$,
including the part that passes through the lags in between. The PACF at lag $k$ is the correlation
that is left *after* controlling for $r_{t-1},\dots,r_{t-k+1}$: the direct link only.

**How to estimate it** (lab instructions): one separate regression per lag $k$,
$$r_t = c + \phi_{k1}\,r_{t-1} + \dots + \phi_{kk}\,r_{t-k} + \varepsilon_t,$$
and keep only the **last** coefficient $\hat\phi_{kk}$. So lag 1 comes from an AR(1), lag 2 from the second coefficient
of an AR(2), and so on. (The five coefficients of a single AR(5) are *not* the PACF: each one there controls for
all the other lags, including the later ones.)

Geometrically, this is the regression idea from the lecture: $\hat\phi_{kk}$ is the correlation between the parts of
$r_t$ and $r_{t-k}$ that are orthogonal to the in-between lags.

**Test**, $H_0: \phi_{kk} = 0$: under white noise, $\hat\phi_{kk} \approx N(0, 1/T)$, so we use the same
$\pm 1.96/\sqrt{T}$ band as for the ACF. The regression's own t-statistic is shown too; it gives almost the same answer.

**Theory for an MA(3).** An invertible MA can be rewritten as an AR($\infty$): $r_t$ depends on *all* its past values,
with weights that shrink geometrically. So the PACF should **not** cut off after lag 3; it should decay gradually
(and can alternate in sign). Cut-off pattern: ACF for MA models, PACF for AR models.

In [12]:
def pacf_by_regressions(x, max_lag=5):
    x = pd.Series(np.asarray(x))
    T = len(x)
    rows = []
    for k in range(1, max_lag + 1):
        X = pd.concat({f"lag{j}": x.shift(j) for j in range(1, k + 1)}, axis=1)
        data = pd.concat([x.rename("y"), X], axis=1).dropna()        # lose k observations
        res = sm.OLS(data["y"], sm.add_constant(data.drop(columns="y"))).fit()
        phi = res.params[f"lag{k}"]
        z = np.sqrt(T) * phi
        rows.append({"lag": k, "pacf": phi, "n used": int(res.nobs),
                     "z = sqrt(T)*pacf": z, "p-value (z)": 2*(1-stats.norm.cdf(abs(z))),
                     "t (OLS)": res.tvalues[f"lag{k}"], "p-value (OLS)": res.pvalues[f"lag{k}"],
                     "CI low": phi - 1.96/np.sqrt(T), "CI high": phi + 1.96/np.sqrt(T)})
    return pd.DataFrame(rows).set_index("lag")

pacf_r = pacf_by_regressions(r)
pacf_r.round(4)

,pacf,n used,z = sqrt(T)*pacf,p-value (z),t (OLS),p-value (OLS),CI low,CI high
lag,,,,,,,,
1,0.2230,996,7.0398,0.0000,7.2076,0.000,0.1609,0.2850
2,-0.1689,995,-5.3324,0.0000,-5.3952,0.000,-0.2310,-0.1068
3,0.1670,994,5.2715,0.0000,5.3232,0.000,0.1049,0.2290
4,-0.0723,993,-2.2824,0.0225,-2.2776,0.023,-0.1344,-0.0102
5,0.0532,992,1.6794,0.0931,1.6714,0.095,-0.0089,0.1153


In [13]:
# Cross-check with statsmodels' built-in PACF (method="ols" runs the same regressions)
from statsmodels.tsa.stattools import pacf
pd.Series(pacf(r, nlags=5, method="ols")[1:], index=range(1, 6), name="statsmodels pacf (ols)").round(4)

1    0.2230
2   -0.1689
3    0.1670
4   -0.0723
5    0.0532
Name: statsmodels pacf (ols), dtype: float64

In [14]:
correlogram(pacf_r, len(r), "Partial autocorrelations of r<sub>t</sub>", col="pacf")

## Step 5 — Exercise 1(d): Theoretical ACF of the MA(3) vs. the sample ACF

**Where the formulas come from.** Write $r_t - \mu = \theta_0 e_t + \theta_1 e_{t-1} + \theta_2 e_{t-2} + \theta_3 e_{t-3}$ with $\theta_0 = 1$.
Because the shocks are uncorrelated, $\text{Cov}(r_t, r_{t-k})$ only picks up products of the **shocks the two returns share**:

$$\gamma_k = \sigma_e^2 \sum_{j=0}^{3-k} \theta_j\,\theta_{j+k}, \qquad \gamma_0 = \sigma_e^2\,(1 + \theta_1^2 + \theta_2^2 + \theta_3^2)$$

- Lag 1: $r_t$ and $r_{t-1}$ share $e_{t-1}, e_{t-2}, e_{t-3}$, giving $\theta_0\theta_1 + \theta_1\theta_2 + \theta_2\theta_3$.
- Lag 2: they share $e_{t-2}, e_{t-3}$, giving $\theta_0\theta_2 + \theta_1\theta_3$.
- Lag 3: they share only $e_{t-3}$, giving $\theta_0\theta_3$.
- Lag $\ge 4$: nothing shared, so $\rho_k = 0$.

Dividing by $\gamma_0$, the $\sigma_e^2$ cancels and we get the lab's formulas for $\rho_1, \rho_2, \rho_3$.

In [15]:
th1, th2, th3 = theta
denom = 1 + th1**2 + th2**2 + th3**2

rho_theory = pd.Series({
    1: (th1 + th1*th2 + th2*th3) / denom,
    2: (th2 + th1*th3) / denom,
    3: th3 / denom,
    4: 0.0,
    5: 0.0,
}, name="theoretical")

# Check: the general shared-shocks formula gives the same numbers
th = np.array([1, th1, th2, th3])
assert np.allclose([th[:4-k] @ th[k:] / (th @ th) for k in (1, 2, 3)], rho_theory[[1, 2, 3]])

compare = pd.DataFrame({"theoretical": rho_theory, "sample": acf_r["rho_hat"]})
compare["sample - theoretical"] = compare["sample"] - compare["theoretical"]
compare.index.name = "lag"
compare.round(4)

,theoretical,sample,sample - theoretical
lag,,,
1,0.1930,0.2229,0.0299
2,-0.1491,-0.1110,0.0381
3,0.0877,0.0882,0.0005
4,0.0000,0.0207,0.0207
5,0.0000,-0.0032,-0.0032


In [16]:
fig = go.Figure()
fig.add_bar(x=compare.index, y=compare["theoretical"], name="Theoretical (MA(3))", marker_color=GREY,
            hovertemplate="lag %{x}<br>theoretical = %{y:.4f}<extra></extra>")
fig.add_bar(x=compare.index, y=compare["sample"], name="Sample", marker_color=BLUE,
            hovertemplate="lag %{x}<br>sample = %{y:.4f}<extra></extra>")
band = 1.96 / np.sqrt(len(r))
for s in (1, -1):
    fig.add_hline(y=s*band, line=dict(color=ORANGE, dash="dash", width=1))
fig.add_hline(y=0, line=dict(color="#3a3a37", width=1))
fig.update_layout(title=f"Theoretical vs. sample ACF of r<sub>t</sub><br><sup>dashed lines: ±1.96/√T = ±{band:.3f}</sup>",
                  xaxis=dict(title="lag k", tickmode="linear", dtick=1), yaxis_title="autocorrelation",
                  barmode="group", bargap=0.35, bargroupgap=0.08, height=380,
                  legend=dict(orientation="h", y=1.02, x=1, xanchor="right", yanchor="bottom"))
fig.show()

## Step 6 — Exercise 1(e): Variance ratio test, VR(5)

**Idea.** Under a random walk the increments are uncorrelated, so variances add (Pythagoras):
the variance of a 5-period return is exactly 5 times the 1-period variance. The variance ratio checks this:

$$VR(q) = \frac{\text{Var}[r_t(q)]}{q\,\text{Var}[r_t]} = \frac{\hat\sigma_c^2(q)}{\hat\sigma_a^2}, \qquad r_t(q) = r_t + r_{t-1} + \dots + r_{t-q+1}$$

- $VR > 1$: multi-period returns are *more* variable than the RW benchmark → positive autocorrelation dominates (momentum).
- $VR < 1$: *less* variable → negative autocorrelation dominates (mean reversion).

**Implementation** (lab formulas, overlapping $q$-period returns):

$$\hat\sigma_c^2 = \frac{1}{m}\sum_{t=q}^{T}\big(r_t(q) - q\hat\mu\big)^2, \qquad m = q(T-q+1)\Big(1-\frac{q}{T}\Big)$$

The divisor $m$ (instead of just the number of terms) corrects for the bias that comes from overlapping windows
sharing observations.

**Test**, $H_0$: random walk. $\mu_r(q) = VR(q) - 1$ and
$$\mu_r(q) \sim N\!\left(0,\; \frac{2(2q-1)(q-1)}{3qT}\right) \quad\Rightarrow\quad z = \frac{\mu_r(q)}{\sqrt{\text{Var}[\mu_r(q)]}} \sim N(0,1)$$

In [17]:
def variance_ratio(x, q=5):
    x = pd.Series(np.asarray(x))
    T = len(x)
    mu = x.mean()

    sigma2_a = x.var()                                  # variance of one-period returns
    x_q = x.rolling(window=q).sum().dropna()            # overlapping q-period returns, t = q..T
    m = q * (T - q + 1) * (1 - q / T)                   # bias-adjusted divisor
    sigma2_c = ((x_q - q * mu) ** 2).sum() / m          # = var(q-period return) / q

    VR = sigma2_c / sigma2_a
    M = VR - 1                                          # mu_r(q): zero under a random walk
    var_M = 2 * (2*q - 1) * (q - 1) / (3 * q * T)
    z = M / np.sqrt(var_M)
    return pd.Series({"T": T, "q": q, "n of q-period returns": len(x_q), "m": m,
                      "sigma2_a": sigma2_a, "sigma2_c": sigma2_c, "VR(q)": VR, "mu_r(q) = VR - 1": M,
                      "Var[mu_r(q)]": var_M, "z": z, "p-value": 2 * (1 - stats.norm.cdf(abs(z)))})

vr_r = variance_ratio(r, q=5)
vr_r.round(4)

T                         997.0000
q                           5.0000
n of q-period returns     993.0000
m                        4940.1003
sigma2_a                    1.1747
sigma2_c                    1.5350
VR(q)                       1.3067
mu_r(q) = VR - 1            0.3067
Var[mu_r(q)]                0.0048
z                           4.4196
p-value                     0.0000
dtype: float64

In [18]:
# Where VR comes from: VR(q) ~ 1 + 2 * sum_{k=1}^{q-1} (1 - k/q) * rho_k
q = 5
weights = pd.Series({k: 2 * (1 - k/q) for k in range(1, q)}, name="weight 2(1-k/q)")
decomp = pd.DataFrame({"rho_hat": acf_r["rho_hat"].loc[1:q-1], "weight": weights})
decomp["contribution"] = decomp["rho_hat"] * decomp["weight"]
print(f"1 + sum of contributions = {1 + decomp['contribution'].sum():.4f}   vs   VR(5) = {vr_r['VR(q)']:.4f}")
decomp.round(4)

1 + sum of contributions = 1.3022   vs   VR(5) = 1.3067


,rho_hat,weight,contribution
1,0.2229,1.6,0.3566
2,-0.1110,1.2,-0.1332
3,0.0882,0.8,0.0706
4,0.0207,0.4,0.0083


# Exercise 2 — Returns equal to $u_t$

Now the return series is $u_t$ itself, so no observations are lost: $T = 1000$.
All the functions from Exercise 1 are reused unchanged.

## Step 7 — Exercise 2(a): Is $u_t$ normally distributed?

In [19]:
u = df["ut"]
print(f"T = {len(u)}")
normality_tests(u, "u").round(4)

T = 1000


,statistic,test stat,p-value
u: skewness (Z),-0.2563,-3.3083,0.0009
u: excess kurtosis (Z),1.8271,11.7937,0.0000
"u: Jarque-Bera (chi2, 2 df)",150.0367,150.0367,0.0000


In [20]:
histogram_vs_normal(u, "u<sub>t</sub>", nbins=50)

In [21]:
from plotly.subplots import make_subplots

def qq_plot(series_dict):
    """Q-Q plots against the normal: sorted standardized data vs. normal quantiles, one panel per series."""
    fig = make_subplots(rows=1, cols=len(series_dict), subplot_titles=list(series_dict),
                        horizontal_spacing=0.1)
    # one common scale for all panels, so the series are directly comparable
    lim = max(np.abs((x - x.mean()) / x.std()).max() for x in series_dict.values()) + 0.3
    for i, (name, x) in enumerate(series_dict.items(), start=1):
        z = np.sort((x - x.mean()) / x.std())                         # standardized, sorted
        p = (np.arange(1, len(z) + 1) - 0.5) / len(z)                  # plotting positions
        q = stats.norm.ppf(p)                                          # normal quantiles
        fig.add_scatter(x=q, y=z, mode="markers", marker=dict(color=BLUE, size=5, opacity=0.6),
                        hovertemplate="normal quantile %{x:.2f}<br>data (in sd) %{y:.2f}<extra></extra>",
                        showlegend=False, row=1, col=i)
        fig.add_scatter(x=[-lim, lim], y=[-lim, lim], mode="lines", line=dict(color=ORANGE, width=1.5),
                        hoverinfo="skip", showlegend=False, row=1, col=i)
        fig.update_xaxes(title_text="normal quantile", range=[-lim, lim], zeroline=False, row=1, col=i)
        fig.update_yaxes(title_text="data quantile (sd units)" if i == 1 else None, range=[-lim, lim], row=1, col=i)
    fig.update_layout(title="Q-Q plots against the normal distribution<br>"
                            "<sup>on the orange line = normal · bending away at the ends = fatter tails than normal</sup>",
                      height=460, margin=dict(t=110))
    fig.show()

qq_plot({"r<sub>t</sub> (MA(3) of e<sub>t</sub>)": r, "u<sub>t</sub>": u})

## Step 8 — Exercise 2(b)–(d): Serial dependence in $u_t$

Same tools as for $r_t$: sample ACF with individual and Ljung–Box tests, PACF from separate AR(k) regressions,
and the VR(5) test. Here $T = 1000$.

In [22]:
acf_u, lb_u = acf_tests(u, "u")
print(f"T = {len(u)},  1.96/sqrt(T) = {1.96/np.sqrt(len(u)):.4f}")
display(acf_u.round(4))
lb_u.round(4)


T = 1000,  1.96/sqrt(T) = 0.0620


,rho_hat,rho (gamma_k/gamma_0),z = sqrt(T)*rho,p-value,CI low,CI high
lag,,,,,,
1,0.0471,0.0471,1.4908,0.1360,-0.0148,0.1091
2,-0.0490,-0.0490,-1.5488,0.1214,-0.1110,0.0130
3,0.0123,0.0123,0.3888,0.6974,-0.0497,0.0743
4,-0.0278,-0.0278,-0.8798,0.3790,-0.0898,0.0342
5,-0.0544,-0.0543,-1.7204,0.0854,-0.1164,0.0076


Q          8.5487
df         5.0000
p-value    0.1285
Name: Ljung-Box u, dtype: float64

In [23]:
correlogram(acf_u, len(u), "Correlogram of u<sub>t</sub>")

In [24]:
pacf_u = pacf_by_regressions(u)
display(pacf_u.round(4))
correlogram(pacf_u, len(u), "Partial autocorrelations of u<sub>t</sub>", col="pacf")

,pacf,n used,z = sqrt(T)*pacf,p-value (z),t (OLS),p-value (OLS),CI low,CI high
lag,,,,,,,,
1,0.0472,999,1.4912,0.1359,1.4903,0.1365,-0.0148,0.1091
2,-0.0513,998,-1.6227,0.1047,-1.6204,0.1055,-0.1133,0.0107
3,0.0172,997,0.5441,0.5864,0.5418,0.5881,-0.0448,0.0792
4,-0.0320,996,-1.0107,0.3122,-1.0058,0.3148,-0.0939,0.0300
5,-0.0502,995,-1.5866,0.1126,-1.5799,0.1145,-0.1122,0.0118


In [25]:
vr_u = variance_ratio(u, q=5)
vr_u.round(4)

T                        1000.0000
q                           5.0000
n of q-period returns     996.0000
m                        4955.1000
sigma2_a                    1.1938
sigma2_c                    1.2211
VR(q)                       1.0228
mu_r(q) = VR - 1            0.0228
Var[mu_r(q)]                0.0048
z                           0.3290
p-value                     0.7421
dtype: float64

In [26]:
summary = pd.DataFrame({
    "r_t": {"JB stat": normality_tests(r, "r").iloc[2, 1], "JB p": normality_tests(r, "r").iloc[2, 2],
            "Ljung-Box Q(5)": lb_r["Q"], "LB p": lb_r["p-value"],
            "# significant ACF lags (1-5)": int((acf_r["p-value"] < 0.05).sum()),
            "VR(5)": vr_r["VR(q)"], "VR z": vr_r["z"], "VR p": vr_r["p-value"]},
    "u_t": {"JB stat": normality_tests(u, "u").iloc[2, 1], "JB p": normality_tests(u, "u").iloc[2, 2],
            "Ljung-Box Q(5)": lb_u["Q"], "LB p": lb_u["p-value"],
            "# significant ACF lags (1-5)": int((acf_u["p-value"] < 0.05).sum()),
            "VR(5)": vr_u["VR(q)"], "VR z": vr_u["z"], "VR p": vr_u["p-value"]},
})
summary.round(4)

,r_t,u_t
JB stat,1.4860,150.0367
JB p,0.4757,0.0000
Ljung-Box Q(5),70.2292,8.5487
LB p,0.0000,0.1285
# significant ACF lags (1-5),3.0000,0.0000
VR(5),1.3067,1.0228
VR z,4.4196,0.3290
VR p,0.0000,0.7421


In [27]:
# ============================================================
# EXPORT: tables (.tex) and figures (.pdf) for the LaTeX report
# Run this last, after all other cells. Needs: pip install kaleido jinja2
# ============================================================
from pathlib import Path

OUT = Path("report")
(OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

def save_tex(table, name, index=True):
    """Write a booktabs tabular (no caption: captions live in main.tex)."""
    n_cols = table.shape[1] + (table.index.nlevels if index else 0)
    table.to_latex(OUT / "tables" / f"{name}.tex", escape=False, float_format="%.3f", index=index,
                   column_format="l" + "r" * (n_cols - 1), multicolumn_format="c")

def pfmt(p):
    return "$<0.001$" if p < 0.001 else f"{p:.3f}"

# --- Table 1: normality tests, both series --------------------------------------
def norm_rows(x, name):
    t = normality_tests(x, name)
    return [t.iloc[0, 0], t.iloc[0, 1], pfmt(t.iloc[0, 2]),
            t.iloc[1, 0], t.iloc[1, 1], pfmt(t.iloc[1, 2]),
            t.iloc[2, 1], pfmt(t.iloc[2, 2])]

tab_norm = pd.DataFrame(
    [norm_rows(r, "r"), norm_rows(u, "u")],
    index=[f"$r_t$ ($T={len(r)}$)", f"$u_t$ ($T={len(u)}$)"],
    columns=pd.MultiIndex.from_tuples([
        ("Skewness", "$S$"), ("Skewness", "$Z_S$"), ("Skewness", "$p$"),
        ("Excess kurtosis", "$K$"), ("Excess kurtosis", "$Z_K$"), ("Excess kurtosis", "$p$"),
        ("Jarque--Bera", "$JB$"), ("Jarque--Bera", "$p$")]))
save_tex(tab_norm, "tab_normality")

# --- Tables 2-3: ACF / PACF by lag -------------------------------------------------
def acf_pacf_table(acf_tab, pacf_tab, theory=None):
    t = pd.DataFrame({"Lag $k$": acf_tab.index}, index=acf_tab.index)
    if theory is not None:
        t["$\\rho_k$ (theory)"] = theory
    t["$\\hat\\rho_k$"] = acf_tab["rho_hat"]
    t["$z$"] = acf_tab["z = sqrt(T)*rho"]
    t["$p$"] = acf_tab["p-value"].map(pfmt)
    t["$\\hat\\phi_{kk}$"] = pacf_tab["pacf"]
    t["$z$ "] = pacf_tab["z = sqrt(T)*pacf"]          # trailing space: pandas needs unique names
    t["$p$ "] = pacf_tab["p-value (z)"].map(pfmt)
    return t

save_tex(acf_pacf_table(acf_r, pacf_r, rho_theory), "tab_acf_r", index=False)
save_tex(acf_pacf_table(acf_u, pacf_u), "tab_acf_u", index=False)

# --- Table 4: joint tests (Ljung-Box and VR(5)), both series ---------------------
tab_joint = pd.DataFrame({
    "$r_t$": [lb_r["Q"], pfmt(lb_r["p-value"]), vr_r["VR(q)"], vr_r["mu_r(q) = VR - 1"], vr_r["z"], pfmt(vr_r["p-value"])],
    "$u_t$": [lb_u["Q"], pfmt(lb_u["p-value"]), vr_u["VR(q)"], vr_u["mu_r(q) = VR - 1"], vr_u["z"], pfmt(vr_u["p-value"])],
}, index=["Ljung--Box $Q_5$", "\\quad $p$ ($\\chi^2_5$)", "$VR(5)$", "$\\mu_r(5) = VR(5)-1$",
          "\\quad $z$", "\\quad $p$ ($N(0,1)$)"])
save_tex(tab_joint, "tab_joint")

# --- Key numbers as LaTeX macros, to quote in the text (e.g. \VRr) -------------------
macros = {
    "Tr": len(r), "Tu": len(u), "band": f"{1.96/np.sqrt(len(r)):.3f}",
    "JBr": f"{tab_norm.iloc[0, 6]:.2f}", "JBu": f"{tab_norm.iloc[1, 6]:.2f}",
    "Qr": f"{lb_r['Q']:.2f}", "Qu": f"{lb_u['Q']:.2f}",
    "VRr": f"{vr_r['VR(q)']:.3f}", "VRu": f"{vr_u['VR(q)']:.3f}",
    "zVRr": f"{vr_r['z']:.2f}", "zVRu": f"{vr_u['z']:.2f}",
}
(OUT / "tables" / "values.tex").write_text(
    "\n".join(f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in macros.items()) + "\n")

# --- Figures: re-run the plotting functions, but capture the figures instead of showing them
captured = []
_show = go.Figure.show
go.Figure.show = lambda self, *a, **k: captured.append(self)
try:
    correlogram(acf_r, len(r), "Sample ACF of r<sub>t</sub>");                        f_acf_r = captured[-1]
    correlogram(pacf_r, len(r), "PACF of r<sub>t</sub>", col="pacf");                 f_pacf_r = captured[-1]
    correlogram(acf_u, len(u), "Sample ACF of u<sub>t</sub>");                        f_acf_u = captured[-1]
    correlogram(pacf_u, len(u), "PACF of u<sub>t</sub>", col="pacf");                 f_pacf_u = captured[-1]
    histogram_vs_normal(u, "u<sub>t</sub>", nbins=50);                                f_hist_u = captured[-1]
    qq_plot({"r<sub>t</sub>": r, "u<sub>t</sub>": u});                               f_qq = captured[-1]
finally:
    go.Figure.show = _show            # always restore normal plotting

# Theoretical vs sample ACF (step 5 chart, rebuilt here)
f_theory = go.Figure()
f_theory.add_bar(x=compare.index, y=compare["theoretical"], name="Theoretical (MA(3))", marker_color=GREY)
f_theory.add_bar(x=compare.index, y=compare["sample"], name="Sample", marker_color=BLUE)
for s in (1, -1):
    f_theory.add_hline(y=s * 1.96 / np.sqrt(len(r)), line=dict(color=ORANGE, dash="dash", width=1))
f_theory.add_hline(y=0, line=dict(color="#3a3a37", width=1))
f_theory.update_layout(title="Theoretical vs. sample ACF of r<sub>t</sub>", barmode="group", height=380,
                       xaxis=dict(title="lag k", dtick=1), yaxis_title="autocorrelation",
                       legend=dict(orientation="h", y=1.02, x=1, xanchor="right", yanchor="bottom"))

figs = {"fig_acf_r": f_acf_r, "fig_pacf_r": f_pacf_r, "fig_theory_r": f_theory,
        "fig_acf_u": f_acf_u, "fig_pacf_u": f_pacf_u, "fig_hist_u": f_hist_u, "fig_qq": f_qq}
half_width = {"fig_acf_r", "fig_pacf_r", "fig_acf_u", "fig_pacf_u", "fig_hist_u", "fig_qq"}
for name, f in figs.items():
    # In the PDF the LaTeX caption replaces the chart title; larger font so text survives the scaling
    f.update_layout(title=None, margin=dict(t=30 if name != "fig_qq" else 45, l=60, r=15, b=50), font_size=15)
    w, h = (560, 400) if name in half_width else (800, 380)
    f.write_image(OUT / "figures" / f"{name}.pdf", width=w, height=h)

print("Written to", OUT.resolve())
for p in sorted(OUT.rglob("*.*")):
    print("  ", p.relative_to(OUT))

Written to C:\Users\Vicente\Desktop\Lund - MSc Economics\ML - From a Regression perspective\ml-course\Empirical Finance\report
   figures\fig_acf_r.pdf
   figures\fig_acf_u.pdf
   figures\fig_hist_u.pdf
   figures\fig_pacf_r.pdf
   figures\fig_pacf_u.pdf
   figures\fig_qq.pdf
   figures\fig_theory_r.pdf
   main.tex
   tables\tab_acf_r.tex
   tables\tab_acf_u.tex
   tables\tab_joint.tex
   tables\tab_normality.tex
   tables\values.tex
